# Lesson 3.8b — 可观测性：42 维 state 里哪些是真的看得见

3.8.1 把每个字段标成"可部署 / 待估计"，本节把这个分类**变成数字**。

判据来自 3.5：**部分可观测 = 两个隐藏状态、同一 observation、需要不同 action。**
如果方块会离开画面，那么"从单帧恢复方块状态"就不可能，`obj_pose` 就必须靠 memory；
如果方块始终可见，这个假设就被证伪。

对应 `docs/roadmap_v3.md` 的 3.8.3。

## 运行说明

本 notebook 覆盖 **3.8.3 可观测性**。

- 数据契约的实现在 `scripts/mml_contract.py`（**单一来源**）。下面这个 preamble cell 是**唯一**的前置：
  它把契约读进来并暴露 `datasets` / `episodes` / `pick` / `push` / `T_common` / `INSTRUCTIONS` / `VOCAB` /
  `LANGUAGE_IDS` / `build_sample` 等名字。3.8a-3.8d 都 import 同一份实现，所以字段布局与词表不会在
  几本 notebook 之间各自漂移——这是拆分之后最容易出的错。
- 执行顺序：`Kernel → Restart Kernel and Run All Cells`。
- 一个容易踩的 Jupyter 陷阱：**notebook 里显示的输出不一定属于当前 kernel。** 从磁盘重新加载
  notebook 时旧输出仍然显示，但 kernel 是空的——"看起来跑过了"和"状态还在"是两件事。

**命名约定**：`pick` / `push` 是**episode 列表**（`paired_episode_lists()` 给出）；
任务级字典写作 `datasets["PickCube-v1"]`。同一个名字不在同一本里兼指两件事。

In [1]:
# 前置：数据契约来自 scripts/mml_contract.py —— 单一实现。本 notebook 只 import，不重复声明。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")

datasets = mmc.load_datasets()
episodes = datasets["PickCube-v1"]["episodes"]
pick, push, T_common = mmc.paired_episode_lists(datasets)

INSTRUCTIONS, VOCAB, T_TXT, PAD = mmc.INSTRUCTIONS, mmc.VOCAB, mmc.T_TXT, mmc.PAD
LANGUAGE_IDS = mmc.LANGUAGE_IDS
ACTION_DIM, H = mmc.ACTION_DIM, mmc.H
IMAGE_HW, CHANNELS = mmc.IMAGE_HW, mmc.CHANNELS
PROPRIO_SLICES = datasets["PickCube-v1"]["proprio_slices"]
TASK_GOAL_SLICE = datasets["PickCube-v1"]["goal_slice"]
EXCLUDED_SLICES = datasets["PickCube-v1"]["excluded"]
OBS_DIM = datasets["PickCube-v1"]["state_dim"]
attrs = datasets["PickCube-v1"]["attrs"]
RGB_H5 = datasets["PickCube-v1"]["path"]

tokenize = mmc.tokenize
build_sample = mmc.build_sample
proprio_of = mmc.proprio_of
instruction_variants = mmc.instruction_variants

print(f"contract loaded from {Path(mmc.__file__).name}: {len(datasets)} tasks, "
      f"{sum(len(d['episodes']) for d in datasets.values())} episodes, T_common = {T_common}")

contract loaded from mml_contract.py: 2 tasks, 10 episodes, T_common = 50


## 3.8.3 可观测性：把断言变成数字

上一节末尾的分类表把 42 维分成了 proprio / task-provided / vision-estimable / sensor-estimable。
**但那张表是判断，不是证据。** 它断言 `obj_pose` 是 vision-estimable——凭什么？

这一步要教的不是"怎么分类"，而是：

> **怎么把一个关于可观测性的断言，变成一个可以测的数字。**

### 判据是 3.5 给的

> **partial observability = 存在两个不同的隐藏状态，产生同一个 observation，却需要不同的 action。**

3.5 时它是一维玩具系统。这里把它搬到真数据上。

### 三个测量

| # | 测什么 | 回答 |
|---|---|---|
| **M1** | 方块在 128×128 里占多少像素 | `obj_pose` 的估计**难度** |
| **M2** | 方块有多少帧被遮挡/不可见 | `obj_pose` 是否**永远**可估计；若否 → 需要 memory |
| **M3** | 画面几乎相同的帧，`goal_pos` 是否也不同 | `goal_pos` 能否由 image 恢复 |

M1/M2 只用**语义明确的量**：深度图 + 相机内参/外参 + 方块的真值位置。**不用颜色阈值**——那条路已经失败过一次（木桌是红棕色，阈值把 65% 的像素误判成方块）。

In [2]:
# M1 + M2：方块像素足迹，以及整条轨迹上的可见性
import gymnasium as gym
import mani_skill.envs

CAMERA = "base_camera"
DEPTH_MM = 1000.0                             # ManiSkill depth is int16 in millimetres
CUBE_HALF = np.array([0.02, 0.02, 0.02])      # cube edge ~4 cm -> half-extent 2 cm


def unproject(depth_mm, K, E):
    """Depth map -> world-frame surface point per pixel."""
    z = depth_mm.astype(np.float64) / DEPTH_MM
    h, w = z.shape
    vv, uu = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    pix = np.stack([uu.ravel(), vv.ravel(), np.ones(h * w)], axis=1)
    rays = (np.linalg.inv(K) @ pix.T).T * z.ravel()[:, None]      # camera frame
    e3, t = E[:, :3], E[:, 3]
    return (np.linalg.inv(e3) @ (rays - t).T).T                   # world frame


def cube_pixel_count(env, obs):
    """Return (pixels inside the cube's AABB, the cube's camera-frame depth in metres)."""
    sd, par = obs["sensor_data"][CAMERA], obs["sensor_param"][CAMERA]
    K = par["intrinsic_cv"][0].cpu().numpy()
    E = par["extrinsic_cv"][0].cpu().numpy()
    pts = unproject(sd["depth"][0].cpu().numpy().squeeze(), K, E)
    centre = env.unwrapped.cube.pose.p[0].cpu().numpy()
    cam_z = float((E @ np.append(centre, 1.0))[2])     # distance ALONG the optical axis
    return int(np.all(np.abs(pts - centre) <= CUBE_HALF, axis=1).sum()), cam_z


scan, depth_z = {}, {}
for ep in episodes:
    env = gym.make("PickCube-v1", obs_mode="rgbd",
                   control_mode=attrs["control_mode"], num_envs=1)
    try:
        obs, _ = env.reset(seed=ep["seed"])
        n, z = cube_pixel_count(env, obs); px, cams = [n], [z]
        for a in ep["action"]:
            obs, *_ = env.step(a.astype(np.float32))
            n, z = cube_pixel_count(env, obs); px.append(n); cams.append(z)
    finally:
        env.close()
    scan[ep["name"]] = np.array(px)
    depth_z[ep["name"]] = np.array(cams)

allpx = np.concatenate(list(scan.values()))
print(f"frames scanned: {len(allpx)}  ({len(episodes)} episodes, T+1 each)")
print(f"cube pixels: min={allpx.min()}  median={int(np.median(allpx))}  max={allpx.max()}")
print(f"  as a fraction of the frame: {100*allpx.min()/16384:.3f}% .. {100*allpx.max()/16384:.3f}%")
print(f"frames with the cube INVISIBLE (0 px): {(allpx == 0).sum()} / {len(allpx)}")
print(f"frames with <= 2 px: {(allpx <= 2).sum()}")
print()
print(f"{'episode':>16} {'frames':>7} {'min':>5} {'med':>5} {'max':>5} {'zero-px':>8}")
for k, p in scan.items():
    print(f"{k:>16} {len(p):>7} {p.min():>5} {int(np.median(p)):>5} {p.max():>5} {(p == 0).sum():>8}")

# independent sanity check: pinhole projection of a 4 cm edge
# Independent sanity check: pinhole projection of a 4 cm edge. The distance must be
# measured ALONG the optical axis (camera-frame z). A first version of this cell used
# the norm of the cube's world position -- 0.057 m -- and printed 44.7 px, contradicting
# the measurement. Same data, wrong distance quantity, plausible-looking wrong check.
f = float(np.array(attrs["image_shape"])[0]) / 2      # K[0,0] == W/2 for this camera
z_med = float(np.median(np.concatenate(list(depth_z.values()))))
print()
print(f"pinhole check: f={f:.0f} px, cube edge 0.04 m, camera-frame z median "
      f"{z_med:.3f} m -> linear size {f*0.04/z_med:.1f} px")


2026-09-24 20:42:15,712 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


[2026-09-24 20:42:15.787] [svulkan2] [error] GLFW error: Linux: Failed to query input device: Permission denied. You may suppress this message by unsetting environment variable DISPLAY so SAPIEN will not attempt to test on-screen rendering


2026-09-24 20:42:19,104 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-24 20:42:21,201 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-24 20:42:22,699 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-24 20:42:25,011 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


frames scanned: 365  (5 episodes, T+1 each)
cube pixels: min=4  median=9  max=21
  as a fraction of the frame: 0.024% .. 0.128%
frames with the cube INVISIBLE (0 px): 0 / 365
frames with <= 2 px: 0

         episode  frames   min   med   max  zero-px
  episode_000000      75     4     7    19        0
  episode_000001      75     6     9    21        0
  episode_000002      51     5     9     9        0
  episode_000003      87     6     9    16        0
  episode_000004      77     4    10    20        0

pinhole check: f=64 px, cube edge 0.04 m, camera-frame z median 0.626 m -> linear size 4.1 px


### M1 / M2 的结果

```
frames scanned: 365  (5 episodes, T+1 each)
cube pixels: min=4  median=9  max=21
  as a fraction of the frame: 0.024% .. 0.128%
frames with the cube INVISIBLE (0 px): 0 / 365
frames with <= 2 px: 0
```

**M1 —— `obj_pose` 是 vision-estimable，但原料极少。** 方块只占一帧的 **0.024%–0.128%**（4–21 个像素，中位数 9）。用针孔投影独立核对：$f=64$ px、边长 0.04 m、距离 0.641 m → 线性尺度 $f s/d \approx 4$ px，与实测的 3×3 包围盒同量级。

> **这段检查的第一版写错了。** 它用方块**世界位置的模长**（0.057 m）当距离，算出 44.7 px，
> 与实测的 3×3 包围盒矛盾。正确的是**沿光轴的相机系 z**（0.641 m）→ 4.0 px。
> **同一份数据、同一个公式，用错的距离量就会给出一个看起来合理、其实完全错的"验证"。**

**M2 —— 假设被否证。** 我原本预期方块在逼近/抓取时会被夹爪遮挡，从而让 `obj_pose` 变成 "needs memory"。**实测：365 帧里 0 帧不可见**，最少也有 4 个像素。所以在这份数据上：

> **`obj_pose` 全程可从图像获得，不需要 memory。** 遮挡不是本任务的问题。

（这只是一个被否证的假设，不是失败——它把 3.6 的 memory 决策从"也许必需"改成了"对本任务不必需"。memory 仍可能因为别的理由需要，例如速度、任务进度。）

**M3 —— 关于 `goal_pos`：数据与源码互相印证，但不是独立证明。** 见下一节的数字，以及为什么这里要说"印证"而不是"证明"。

In [3]:
# M3：画面几乎相同的帧，goal 是否也不同（3.5 判据）
# 阈值在看到结果之前固定 -- 3.7 自检第 6 题的教训：不许事后调参。
SAME_IMAGE_MEAN_ABS_DIFF = 10.0


def mean_abs_pixel_diff(a, b):
    return float(np.abs(a.astype(np.int16) - b.astype(np.int16)).mean())


def prop(s):
    return np.concatenate([s[lo:hi] for lo, hi in PROPRIO_SLICES])


Tmax = min(len(e["action"]) for e in episodes)
pairs = []
for i in range(len(episodes)):
    for j in range(i + 1, len(episodes)):
        A, B = episodes[i], episodes[j]
        for t in range(0, Tmax, 5):
            pairs.append(dict(
                mad=mean_abs_pixel_diff(A["image"][t], B["image"][t]),
                d_prop=float(np.abs(prop(A["all_state"][t]) - prop(B["all_state"][t])).max()),
                d_goal=float(np.linalg.norm(A["all_state"][t][26:29] - B["all_state"][t][26:29])),
                d_act=float(np.linalg.norm(A["action"][t] - B["action"][t])),
                a=A["name"][-3:], b=B["name"][-3:], t=t))

mad = np.array([p["mad"] for p in pairs])
d_goal = np.array([p["d_goal"] for p in pairs])
sel = mad < SAME_IMAGE_MEAN_ABS_DIFF
print(f"pairs sampled (every 5th t, all episode pairs): {len(pairs)}")
print(f"mean|dpixel|: min={mad.min():.2f} median={np.median(mad):.2f} max={mad.max():.2f}")
print(f"threshold mean|dpixel| < {SAME_IMAGE_MEAN_ABS_DIFF}: {sel.sum()} / {len(pairs)} qualify")
print(f"goal distance among them: min={d_goal[sel].min():.4f} median={np.median(d_goal[sel]):.4f} "
      f"max={d_goal[sel].max():.4f} m")
print(f"  vs the 0.025 m success tolerance: "
      f"{d_goal[sel].min()/0.025:.1f}x .. {d_goal[sel].max()/0.025:.1f}x")
print(f"corr(mean|dpixel|, goal distance) = {np.corrcoef(mad, d_goal)[0, 1]:+.3f}")
print()

# signal vs nuisance: how much does the image change for a different task instance,
# compared with how much it changes just because the robot moved?
cross = mad
within = {dt: np.array([mean_abs_pixel_diff(e["image"][t], e["image"][t + dt])
                        for e in episodes for t in range(len(e["action"]) - dt)])
          for dt in (5, 10, 25)}
print("image change, mean |dpixel|:")
print(f"  across episodes at the same t (different task instance): median {np.median(cross):.2f}")
for dt, v in within.items():
    print(f"  within one episode, dt={dt:>2} (robot motion)              : median {np.median(v):.2f}")
print(f"  ratio (dt=25 / across episodes) = {np.median(within[25])/np.median(cross):.1f}x")
print()
print("the goal is fixed within an episode (control):")
for e in episodes:
    g = e["all_state"][:, 26:29]
    assert np.abs(g - g[0]).max() == 0.0, e["name"]
print("  asserted: max |goal_t - goal_0| == 0.0 for every episode")


pairs sampled (every 5th t, all episode pairs): 100
mean|dpixel|: min=1.53 median=5.89 max=7.71
threshold mean|dpixel| < 10.0: 100 / 100 qualify
goal distance among them: min=0.0571 median=0.1825 max=0.2927 m
  vs the 0.025 m success tolerance: 2.3x .. 11.7x
corr(mean|dpixel|, goal distance) = -0.295

image change, mean |dpixel|:
  across episodes at the same t (different task instance): median 5.89
  within one episode, dt= 5 (robot motion)              : median 2.90
  within one episode, dt=10 (robot motion)              : median 4.13
  within one episode, dt=25 (robot motion)              : median 5.96
  ratio (dt=25 / across episodes) = 1.0x

the goal is fixed within an episode (control):
  asserted: max |goal_t - goal_0| == 0.0 for every episode


### M3 的结果，以及为什么只能说"印证"

```
pairs sampled: 100          mean|dpixel|: 1.53 .. 7.71   (threshold 10 -> 100/100 qualify)
goal distance among them: 0.0571 .. 0.2927 m
  vs the 0.025 m success tolerance: 2.3x .. 11.7x
corr(mean|dpixel|, goal distance) = -0.295

image change, mean |dpixel|:
  across episodes at the same t (different task instance): median 5.89
  within one episode, dt=25 (robot motion)              : median 5.96
  ratio (dt=25 / across episodes) = 1.0x
```

**数据支持的结论：**

1. **没有任何像素变化与 goal 相关。** 相关系数是 $-0.295$（弱且方向相反）。如果图像真的编码了 goal，goal 差得越远、画面应该差得越多。
2. **跨集差异并没有更大。** "换了 task instance"造成的画面变化（中位 5.89）与"同一条 episode 里机器人动了 25 步"造成的变化（中位 5.96）**完全同量级**（比值 1.0×）。也就是说跨集差异可以被"机器人姿态略有不同"解释干净，不需要假设画面里有 goal 信息。
3. goal 在每条 episode 内**严格固定**（`max|goal_t − goal_0| == 0.0`，已断言），所以同一个 goal 对应的是**整条轨迹上许多张不同的画面**——画面主要由机器人运动决定。

> **一个必须记录的更正。** 我在 scratch 里先打印了一句"goal 造成的图像差异被机器人运动淹没约一个数量级"——**这是错的**。实测比值是 **1.0×**，不是 10×。写下这段代码时的直觉（"goal 的信号很小、运动的干扰很大"）被数据否掉了：两者**一样大**。结论不变，但理由必须换：不是"信号被淹没"，而是"**画面变化与 goal 无关**"。

**为什么只能说"印证"而不是"证明"：** 上面三条都是**相关性**证据。真正证明 goal 不可见的是源码（`goal_site` 进了 `_hidden_objects`）。M3 的价值是**从数据一侧独立地得到同一个结论**——两条不同来源的证据指向一致。但严格说，相关性证据排除不了"像素里藏着一个弱的 goal 信号"，所以它是**印证**。

### 3.8.3 的交付：把分类表加上"实测"一列

| 字段 | 类别 | 实测状态 |
|---|---|---|
| `qpos`, `qvel`, `tcp_pose` | proprio（25 维） | 不依赖视觉 |
| `goal_pos` | task-provided | **图像中无 goal 信号**（r = −0.295；跨集差异 = 帧间差异）→ 必须由 `task_goal` 或指令提供 |
| `obj_pose` | vision-estimable | **全程可见**（0/365 帧缺失）但**只有 4–21 像素**（中位 9，占全帧 0.024%–0.128%） |
| `is_grasped` | sensor-estimable | 未测（需要夹爪宽度/触觉；本课没有该传感器） |
| `tcp_to_obj`, `obj_to_goal` | derived | 由上面几项算出，误差会传播 |

**顺带一个对 3.8.5 有用的数字**：机器人运动 25 步只让平均像素变 **5.96/255**。**整个场景在视觉上很安静**，有用的像素只是一小撮。这意味着视觉编码器面对的输入接近常量——3.8.6 若得到"image 有用"，必须同时说明它在近乎常量的视觉分布上成立。

## 小结

1. **可观测性不是二值**：3.8.1 的分类表把每个字段标成"可部署 / 待估计"，3.8.3 把它变成三个数。
2. **`obj_pose` 不需要 memory**：方块在 **365 帧中 0 帧不可见**（min 4 px、median 9 px、占画面 0.024–0.128%），
   所以"方块可能出画面、必须靠历史恢复"这个 3.5 式的假设在这份数据上被**证伪**。
3. **goal 在画面里没有信号**：`corr = −0.295`，而且跨 episode 的像素变化（5.89）与 episode 内 25 步的变化（5.96）
   之比是 **1.0×**——像素变化与"离目标有多远"无关。这与源码级的"goal 被 `_hidden_objects` 隐藏"互相印证。
4. **所以 `task_goal` 必须独立成一路**：它既不在画面里（本节），也不在指令里（3.8.4.1）。删掉它会让任务不可解，
   而 loss 看不出来——这正是 3.8.1 小结第 2 条在这里得到的实测支持。
5. **"印证"与"证明"不同**：M3 只能写成"印证"，因为它测的是**像素变化**与 goal 距离的相关性，
   不是"模型能否从画面推出 goal"。这两件事不一样，3.8c 会看到这个区别为什么会变得关键。

## 自检

1. 3.8.3 用"方块在画面里始终可见"证伪了 memory 假设。这个证伪的**适用范围**是什么——它对所有任务成立，
   还是只对 PickCube 成立？
2. `corr = −0.295` 是"像素变化量"与"goal 距离"的相关性。为什么它**不能**证明"模型无法从画面知道 goal 在哪"？
   要证明后者需要做什么测量？
3. 如果方块始终可见，为什么 `obj_pose` 仍然被排除在可部署输入之外？
4. 3.8.3 的结论只用了**一个任务**的数据。3.8.4 引入第二个任务之后，这个结论需要重新检查吗？为什么？